# 28 孟加拉国 DIA 全量清洗、审计与冻结（V2）

从 MS-DIAL 原始导出重新执行质量筛选、Alignment ID 关联、谱图标准化、异常检查和精确去重。原始导出只读，结果写入版本化 `28_full_v2`。

建议内核：`env_ms`。

In [ ]:
from pathlib import Path
import sys, json
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'pbt_msms').exists(): ROOT=ROOT.parent
assert (ROOT/'src'/'pbt_msms').exists(), '找不到项目根目录'
if str(ROOT/'src') not in sys.path: sys.path.insert(0,str(ROOT/'src'))
print('ROOT =',ROOT)
print('Python =',sys.executable)


In [ ]:
from pbt_msms.bangladesh_dia_external import run_cleaning, paths
summary=run_cleaning(ROOT)
print(json.dumps(summary,ensure_ascii=False,indent=2))


In [ ]:
import pandas as pd
p=paths(ROOT)
rows=[]
for mode in ('Pos','Neg'):
    manifest=json.loads((p['clean']/mode/'28_run_manifest.json').read_text(encoding='utf-8'))
    rows.append({'mode':mode,**manifest['counts'],**manifest['id_integrity']})
display(pd.DataFrame(rows))
assert all(x['all_equal'] for x in rows)
assert summary['status']=='FROZEN_FOR_PBT_INPUT_PREPARATION'
print('PASS：第28步已冻结，可进入第29步。')


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(2,2,figsize=(14,9))
for r,mode in enumerate(('Pos','Neg')):
    table=pd.read_csv(p['clean']/mode/'28_feature_screening.tsv',sep='\t')
    audit=pd.read_csv(p['clean']/mode/'28_spectrum_audit.tsv',sep='\t')
    table.loc[table.final_pass,'adduct'].value_counts().plot.bar(ax=axes[r,0],title=f'{mode} final adducts')
    audit.cleaned_peak_n.plot.hist(bins=50,ax=axes[r,1],title=f'{mode} cleaned peak counts')
plt.tight_layout(); plt.show()
